# ArchVis g9 이후 한정 사전 인덱싱 — CPU
새 Colab CPU 런타임에서 고정 source/input 두 ZIP을 준비합니다. 학습과 독립된 한 번의 job으로 원본 archive를 순차 확장합니다.

Validation pose 관측 1,331 key / 20 sequence가 선정 근거입니다. RGB 위치와 이번 예산 내 발견은 보장하지 않습니다. 최신 pose 목록으로 기존 stereo 102개를 대조합니다.

최초 metadata를 포함한 한도는 body 512MiB / HTTP 4,096회 / 1,200초 / read 1GiB / decode 512MiB / RSS 1GiB입니다. Work·Drive snapshots·회수 ZIP의 논리 합은 768MiB, 32 member씩 최대 32 batch이며 시간 reserve는 30초입니다.

가용 RAM과 임시 disk는 각각 2GiB 이상이어야 합니다. Upload·source/input 해제·의존성 설치·인증/mount는 phase 밖의 별도 비용입니다.

Snapshot을 새 Drive 폴더에 배치별 불변 파일로 저장합니다. 마지막 정상 receipt와 snapshot만 회수 ZIP에 포함하며, 입력 g9와 원본 Drive 파일은 변경하지 않습니다.

공간을 사전 예약하므로 32 batch 전에 bounded_stop할 수 있습니다. 자동 삭제·후속 실행·budget 초기화는 없습니다. 실패·중단 뒤에는 회수 셀만 실행하세요.

Timing은 서로 겹치는 wall/process CPU와 HTTP 호출~headers / raw.read1 소비 시간입니다. 순수 network/gzip 시간을 분리한 값이 아닙니다.

Core index 32MiB / 20,000 member / 4,096 point / snapshot DB 64MiB 상한을 유지합니다. 전체 3TB 완료·GPU·병렬화·학습 release는 이번 범위에 포함하지 않습니다.


In [ ]:
# Root가 커밋 후 source/ready revision·SHA와 새 Drive 폴더를 고정합니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
NOTEBOOK_REVISION = ""
INPUT_ZIP_SHA256 = "e7e46e17921e54afc78c7b2e7cdcdb7cd9bbc511adfd547f47ccbececa4a24f1"
SELECTION_SHA256 = "0a3c216da321bd5b9385cb79de20fb2d3ed2da78a0109177c2cdfd4dd1320ba1"
CONFIG_SHA256 = "1a94db146a28aeddac51fc7cf59e82661358ff15ed3158aeda422de5d0ea7d44"
JOB_ID = "archvis_rgb_preindex_g9_001"
SOURCE_ROOT = "/content/archvis_rgb_preindex_g9_001-source"
INPUT_ROOT = "/content/archvis_rgb_preindex_g9_001-inputs"
WORKSPACE = "/content/archvis_rgb_preindex_g9_001-work"
SNAPSHOT_DIRECTORY = ""  # 새 별도 MyDrive 폴더; 원본/기존 snapshot 폴더 금지
INPUT_SNAPSHOT_DIRECTORY = INPUT_ROOT+"/snapshot"
RECOVERY_ZIP = "/content/archvis_rgb_preindex_g9_001_result.zip"


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", SELECTION_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert SNAPSHOT_DIRECTORY and not Path(WORKSPACE).exists() and not Path(WORKSPACE).is_symlink(), "소비된 job: 준비/scan 재실행 금지"
import psutil,shutil
assert psutil.virtual_memory().available>=2*1024**3,"가용 RAM2GiB 필요"
assert shutil.disk_usage(Path(SOURCE_ROOT).parent).free>=2*1024**3,"가용 disk2GiB 필요"
print("1/2: 고정 source ZIP을 선택하세요.")
source_uploaded = files.upload()
print("2/2: 고정 input ZIP을 선택하세요.")
input_uploaded = files.upload()
def selected(uploaded, pin, label):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, label + " ZIP 누락 또는 고정 SHA 불일치"
    return rows[0]
def inspect_zip(raw, maximum):
    assert len(raw) <= maximum, "압축 ZIP 크기 상한"
    payloads, directories, names = {}, set(), set()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum, "ZIP 해제 상한"
        for item in rows:
            name = item.filename.rstrip("/")
            path = PurePosixPath(name)
            assert name and name == path.as_posix() and not path.is_absolute()
            assert ".." not in path.parts and "\\" not in name and all(":" not in part for part in path.parts), "ZIP 경로 오류"
            assert name.casefold() not in names, "ZIP 중복 경로"
            names.add(name.casefold())
            kind = stat.S_IFMT(item.external_attr >> 16)
            assert kind in (0, stat.S_IFDIR if item.is_dir() else stat.S_IFREG), "ZIP 링크/특수파일 금지"
            for parent in path.parents:
                if str(parent) != ".": directories.add(parent.as_posix())
            data = z.read(item)  # 모든 member CRC를 추출 전에 검사
            assert len(data) == item.file_size
            if item.is_dir(): directories.add(name)
            else: payloads[name] = data
    assert not set(payloads) & directories, "ZIP 파일/폴더 충돌"
    return payloads, directories
# 두 ZIP 모두 검증하기 전에는 source/input/준비 디렉터리를 만들지 않는다.
source_data = inspect_zip(selected(source_uploaded, SOURCE_ZIP_SHA256, "source"), 64*1024**2)
input_data = inspect_zip(selected(input_uploaded, INPUT_ZIP_SHA256, "input"), 64*1024**2)
assert source_data[0]["SOURCE_REVISION.txt"].decode("utf-8").strip() == SOURCE_REVISION
input_config = json.loads(input_data[0]["config.json"])
assert hashlib.sha256(json.dumps(input_config["selection"], ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest() == SELECTION_SHA256
def existing_matches(destination, contents):
    target = Path(destination).absolute()
    for part in (target, *target.parents):
        assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)(), "추출 경로 링크 금지"
    if not target.exists(): return False
    assert target.is_dir(), "추출 대상이 디렉터리가 아님"
    files_seen, dirs_seen = set(), set()
    for count, path in enumerate(target.rglob("*")):
        assert count < 4096 and not path.is_symlink() and not getattr(path, "is_junction", lambda:False)()
        relative = path.relative_to(target).as_posix()
        if path.is_dir(): dirs_seen.add(relative)
        else:
            assert path.is_file() and relative in contents[0], "기존 추출본의 다른 파일"
            expected = contents[0][relative]
            assert path.stat().st_size == len(expected) and path.read_bytes() == expected, "기존 추출본 바이트 불일치"
            files_seen.add(relative)
    assert files_seen == set(contents[0]) and dirs_seen == contents[1], "기존 추출본 파일/폴더 집합 불일치"
    return True
source_path, input_path = Path(SOURCE_ROOT).absolute(), Path(INPUT_ROOT).absolute()
assert not source_path.is_relative_to(input_path) and not input_path.is_relative_to(source_path)
source_exists = existing_matches(SOURCE_ROOT, source_data)
input_exists = existing_matches(INPUT_ROOT, input_data)
def materialize(destination, contents, exists):
    if exists: return  # 정확히 같은 기존 추출본만 읽기 재사용
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=False)
    for name in sorted(contents[1], key=lambda s:len(PurePosixPath(s).parts)):
        (target/name).mkdir(exist_ok=True)
    for name, data in contents[0].items():
        with (target/name).open("xb") as stream: stream.write(data)
materialize(SOURCE_ROOT, source_data, source_exists)
materialize(INPUT_ROOT, input_data, input_exists)
sys.dont_write_bytecode = True
sys.path.insert(0, SOURCE_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "indexed_gzip==1.10.3", "psutil"])
import indexed_gzip
assert indexed_gzip.__version__ == "1.10.3"
del source_uploaded, input_uploaded, source_data, input_data


In [ ]:
# 인증/원본 요청 전 exact input·g9 anchor·profile을 검증합니다.
import sys
from pathlib import Path
for name,module in tuple(sys.modules.items()):
    if name.split(".")[0] in ("rewind_archive","rewind_archive_index","rewind_discovery"):
        loaded=getattr(module,"__file__",None)
        assert loaded and Path(loaded).resolve().is_relative_to(Path(SOURCE_ROOT).resolve()),"이전 코드가 로드됨: 새 CPU 런타임에서 시작하세요"
sys.path.insert(0,SOURCE_ROOT)
from rewind_discovery.preindex import preflight
CONFIG,ARGS=preflight(INPUT_ROOT,CONFIG_SHA256)
assert CONFIG["job_id"]==JOB_ID
print({"job_id":JOB_ID,"profile":CONFIG["profile"],"limits":CONFIG["limits"],"g9":CONFIG["resume"]["anchor"],"baseline":CONFIG["baseline"]})


In [ ]:
# 한 번만 실행. auth 뒤에는 1200초 phase budget에 준비/scan/snapshot을 포함합니다.
from pathlib import Path
assert not Path(WORKSPACE).exists(), "이미 소비한 job: 회수 셀만 실행"
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
import os
mount = Path("/content/drive")
my_drive = mount/"MyDrive"
assert os.path.ismount(mount) and my_drive.is_dir(), "실제 Drive mount 확인 실패"
snapshot_path = Path(SNAPSHOT_DIRECTORY).resolve()
assert snapshot_path.is_relative_to(my_drive.resolve()) and snapshot_path != my_drive.resolve()
assert not snapshot_path.exists(), "새 별도 snapshot 폴더만 허용"
import google.auth
import requests
from google.auth.transport.requests import AuthorizedSession, Request
from rewind_archive.request_policy import validate_no_hidden_retries
from urllib3.util.retry import Retry
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
def session_factory():
    auth_http = requests.Session()
    try:
        auth_http.trust_env = False
        for adapter in auth_http.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        session = AuthorizedSession(credentials, max_refresh_attempts=0, auth_request=Request(session=auth_http))
    except BaseException:
        auth_http.close()
        raise
    close_outer = session.close
    def close_all():
        try:
            close_outer()
        finally:
            auth_http.close()
    session.close = close_all
    try:
        session.trust_env = False
        for adapter in session.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        # 초기 metadata 호출 전 내부 auth transport까지 검사한다.
        validate_no_hidden_retries(session)
    except BaseException:
        session.close()
        raise
    return session
from rewind_discovery.runner import run
report=run(**ARGS,workspace=WORKSPACE,snapshot_directory=SNAPSHOT_DIRECTORY,session_factory=session_factory)
print(json.dumps({k:report.get(k) for k in ("status","failure","body_bytes","requests","after","working_discovered","durable_discovered","last_receipt","timing")},ensure_ascii=False))


In [ ]:
# 독립 회수: input config/인증/session/scan/run 없이 고정 selection digest로 회수합니다.
import hashlib, json, re, sys, zipfile
from pathlib import Path
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SELECTION_SHA256)
target = Path(RECOVERY_ZIP)
if not Path(WORKSPACE).exists():
    target = target.with_name(target.stem.removesuffix("_result") + "_preparation_result" + target.suffix)
pin_path = target.with_suffix(".sha256")
def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        while data := stream.read(65536): h.update(data)
    return h.hexdigest()
if target.exists():
    assert pin_path.is_file() and file_sha(target) == pin_path.read_text().strip()
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info = json.loads(z.read("recovery.json"))
        assert info["job_id"] == JOB_ID and info["selection_sha256"] == SELECTION_SHA256
else:
    assert not pin_path.exists()
    if Path(WORKSPACE).exists():
        # 고정 source와 마지막 정상 receipt를 사용합니다. 원본 요청은 하지 않습니다.
        assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
        sys.dont_write_bytecode = True
        sys.path.insert(0, SOURCE_ROOT)
        from rewind_discovery.recovery import recover
        result = recover(WORKSPACE, SNAPSHOT_DIRECTORY, target, job_id=JOB_ID,
            selection_sha256=SELECTION_SHA256, input_snapshot_directory=INPUT_SNAPSHOT_DIRECTORY,disk_limit=768*1024**2)
    else:
        # 준비 실패/부분 source 상태도 source 모듈 import 없이 stdlib로 회수.
        info = {"job_id":JOB_ID, "selection_sha256":SELECTION_SHA256, "last_receipt":None,
            "report_present":False, "execution_status":"workspace_absent_execution_state_unverified",
            "source_root_present":Path(SOURCE_ROOT).is_dir(), "input_root_present":Path(INPUT_ROOT).is_dir(),
            "archive_network_requested":False, "train_ready":False,
            "source_revision":SOURCE_REVISION, "notebook_revision":NOTEBOOK_REVISION}
        with zipfile.ZipFile(target, "x", zipfile.ZIP_STORED) as z:
            z.writestr("recovery.json", json.dumps(info, ensure_ascii=False, sort_keys=True))
    with pin_path.open("x") as stream: stream.write(file_sha(target)+"\n")
print("회수 ZIP SHA256:", file_sha(target))
files.download(str(target))
